This code does the following:
- Trains RFCML model on manual labels
- Predicts fight labels for new dataset
- Plots ethogram and annotated video

To install the environment:

conda env create -f environment.yml
conda activate QuantBio_env

Files required:
- Ground truth labels, respective videos, trajectories
- New dataset with respective trajectories, raw video

- Expected layout: all GT files are stored directly in ../quantBehData (no gt1/gt2/gt3 subfolders)
  - gt1.mp4, gt1_trajectories.csv, gt1_manual_labeled_fights.csv
  - gt2.mp4, gt2_trajectories.csv, gt2_manual_labeled_fights.csv
  - gt3.mp4, gt3_trajectories.csv, gt3_manual_labeled_fights.csv

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier


In [ ]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score, 
    recall_score,
    f1_score,
    precision_recall_curve,
    roc_curve
)
from sklearn.model_selection import train_test_split
from collections import deque
import cv2
import glob
import QuantBio_functions as functions
import importlib
importlib.reload(functions)
%matplotlib inline

In [ ]:
from pathlib import Path
import requests

RECORD_ID = "18817834"
OUT_DIR = Path.cwd().parent / "quantBehData"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Zenodo record JSON (lists files and download URLs)
api_url = f"https://zenodo.org/api/records/{RECORD_ID}"
record = requests.get(api_url).json()

files = record.get("files", [])
print(f"Found {len(files)} files in Zenodo record {RECORD_ID}")

def download(url, dest: Path):
    if dest.exists():
        print(f"✓ {dest.name} already exists")
        return
    print(f"↓ Downloading {dest.name} ...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print(f"✓ Done: {dest.name}")

for f in files:
    name = f["key"]                 # filename on Zenodo
    url  = f["links"]["self"]       # direct download URL
    download(url, OUT_DIR / name)

print("All downloads complete.")

Combine gt*_manual_labeled_fights with gt*_trajectories into temporal training features (matched by gt-prefix in filenames)

In [ ]:
GT_ROOT = functions.get_quant_beh_data_dir()

temporal_csvs = functions.process_gt_datasets_from_flat_root(
    gt_root=GT_ROOT,
    gt_ids=functions.DEFAULT_GT_IDS,
    fps=60,
    window_size=40,
    skip_missing=False
)

Train RFCML

In [ ]:
GT_ROOT = functions.get_quant_beh_data_dir()

temporal_csvs = functions.list_gt_temporal_csvs_flat(GT_ROOT)

print("Found GT videos:", len(temporal_csvs))
for p in temporal_csvs:
    print(p)

In [ ]:
#split test and train data

np.random.seed(42)

all_indices = np.arange(len(temporal_csvs))
test_idx = np.random.choice(all_indices, size=1, replace=False)
train_idx = np.setdiff1d(all_indices, test_idx)

print("Train indices:", train_idx)
print("Test index:", test_idx)


In [ ]:
train_dfs = []
test_dfs = []

for i, path in enumerate(temporal_csvs):
    df = pd.read_csv(path)

    if i in train_idx:
        train_dfs.append(df)
    else:
        test_dfs.append(df)

train_df = pd.concat(train_dfs, ignore_index=True)
test_df  = pd.concat(test_dfs, ignore_index=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


In [ ]:
X_train = train_df.drop(columns=["frame", "label"])
y_train = train_df["label"]

X_test  = test_df.drop(columns=["frame", "label"])
y_test  = test_df["label"]


In [ ]:
clf = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train, y_train)


In [ ]:
threshold=0.4

y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)[:, 1]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

y_pred_adj = (y_probs >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_adj)
TN, FP, FN, TP = cm.ravel()

FPR = FP / (FP + TN)
FNR = FN / (FN + TP)

print("\n=== THRESHOLDED RESULTS ===")
print(cm)
print(classification_report(y_test, y_pred_adj))

print(f"False Positive Rate: {FPR*100:.2f}%")
print(f"False Negative Rate: {FNR*100:.2f}%")


In [ ]:
# save the RFCML model in ../quantBehData
DATA_ROOT = functions.get_quant_beh_data_dir()
model_save_path = os.path.join(DATA_ROOT, "rfcml_model.joblib")

joblib.dump(
    (clf, X_train.columns.tolist(), threshold),
    model_save_path
)

print("Model saved to:")
print(model_save_path)

Predict fights in a new dataset

In [ ]:
# get temporal features for a new dataset in ../quantBehData
DATA_ROOT = functions.get_quant_beh_data_dir()
dataset_id = "new_dataset"  # expects new_dataset_trajectories.csv in DATA_ROOT

temporal_csv = functions.process_dataset_id_from_flat_root(
    data_root=DATA_ROOT,
    dataset_id=dataset_id,
    fps=60,
    window_size=40,
    require_labels=False
)

In [ ]:
DATA_ROOT = functions.get_quant_beh_data_dir()
dataset_id = "new_dataset"

model_path = os.path.join(DATA_ROOT, "rfcml_model.joblib")
clf, expected_features, threshold = joblib.load(model_path)

print("Model loaded.")
print("Threshold:", threshold)

temporal_csv = os.path.join(DATA_ROOT, f"{dataset_id}_temporal_features.csv")
df_new = pd.read_csv(temporal_csv)

if "frame" not in df_new.columns:
    raise ValueError("Temporal CSV must contain 'frame' column.")

missing = set(expected_features) - set(df_new.columns)
if missing:
    raise ValueError(f"Missing expected features: {missing}")

X_new = df_new[expected_features]
probs = clf.predict_proba(X_new)[:, 1]
preds = (probs >= threshold).astype(int)

df_new["fight_probability"] = probs
df_new["predicted_label"] = preds

output_path = os.path.join(DATA_ROOT, f"{dataset_id}_predictions.csv")
df_new.to_csv(output_path, index=False)

print("Predictions saved to:")
print(output_path)

Visualizations

In [ ]:
DATA_ROOT = functions.get_quant_beh_data_dir()
dataset_id = "new_dataset"
pred_path = os.path.join(DATA_ROOT, f"{dataset_id}_predictions.csv")

df = pd.read_csv(pred_path)

fight_prob = df["fight_probability"].to_numpy()
prob_2d = fight_prob[np.newaxis, :]

cmap = LinearSegmentedColormap.from_list(
    "lightblue_red",
    ["#e6f2ff", "red"]
 )

fig, ax = plt.subplots(figsize=(15, 3))

im = ax.imshow(
    prob_2d,
    aspect="auto",
    cmap=cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest"
 )

ax.set_yticks([])
ax.set_xlabel("Frame")

cbar = plt.colorbar(im, ax=ax, orientation="vertical", pad=0.02)
cbar.set_label("Fight probability", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
DATA_ROOT = functions.get_quant_beh_data_dir()
dataset_id = "new_dataset"

functions.annotate_most_fight_chunk(
    video_path=os.path.join(DATA_ROOT, f"{dataset_id}.mp4"),
    trajectory_csv=os.path.join(DATA_ROOT, f"{dataset_id}_trajectories.csv"),
    predictions_csv=os.path.join(DATA_ROOT, f"{dataset_id}_predictions.csv"),
    output_path=os.path.join(DATA_ROOT, f"{dataset_id}_annotated.mp4"),
    window_minutes=5,
    threshold=0.4,
    trail_length=10
)